# 04 — Evaluate the best run + register to UC

Pick the leader from `03`, run `mlflow.evaluate()` to get a proper evaluation
report (with explainability + per-segment metrics), then **register the model
to Unity Catalog** and assign aliases:

- `@champion` → the production-blessed version (this run)
- `@challenger` → set on the second-best run so the next notebook can compare

In a real lifecycle, `@challenger` is what you'd promote to `@champion` after
shadow traffic + offline eval beats the incumbent. We show both so the demo
includes the full alias dance.

In [1]:
import os
import pandas as pd
import mlflow
from mlflow.tracking import MlflowClient
from dotenv import load_dotenv
load_dotenv()

try:
    spark
except NameError:
    from databricks.connect import DatabricksSession
    spark = DatabricksSession.builder.serverless().getOrCreate()

from databricks.sdk import WorkspaceClient
w = WorkspaceClient()
me = w.current_user.me().user_name

UC_CATALOG = os.getenv("UC_CATALOG", "alexander_booth")
UC_SCHEMA  = os.getenv("UC_SCHEMA",  "hockey_xg_mlflow")
EXPERIMENT = os.getenv("MLFLOW_EXPERIMENT_NAME", f"/Users/{me}/hockey-xg-mlflow")
MODEL_NAME = f"{UC_CATALOG}.{UC_SCHEMA}.xg_model"
FEATURES   = f"{UC_CATALOG}.{UC_SCHEMA}.shots_features"

mlflow.set_tracking_uri("databricks")
mlflow.set_registry_uri("databricks-uc")
exp = mlflow.set_experiment(EXPERIMENT)
client = MlflowClient()
print(f"Experiment: {exp.name}\nModel:      {MODEL_NAME}")

/Users/alexander.booth/miniconda3/envs/demo-env/lib/python3.12/site-packages/pyspark/sql/connect/client/core.py:141: UserWarning: Could not enable SetAllowOversizeProtos, please check the protobuf version.
  warnings.warn("Could not enable SetAllowOversizeProtos, please check the protobuf version.")


Experiment: /Users/alexander.booth@databricks.com/hockey-xg-mlflow
Model:      alexander_booth.hockey_xg_mlflow.xg_model


## Pick the leader

In [2]:
runs = client.search_runs(
    experiment_ids=[exp.experiment_id],
    filter_string="attributes.status = 'FINISHED'",
    run_view_type=1,  # ACTIVE_ONLY — exclude soft-deleted runs
    order_by=["metrics.test_log_loss ASC"],
    max_results=20,
)
assert len(runs) >= 2, "Expected at least 2 finished runs from notebook 03"

champion_run = runs[0]
champ_family = champion_run.data.tags.get("model_family")
# Challenger = best run of a *different* family — keeps the alias dance interesting
challenger_run = next(
    (r for r in runs[1:] if r.data.tags.get("model_family") != champ_family),
    runs[1],
)
print(f"Champion:   {champion_run.data.tags.get('mlflow.runName')}  (log_loss={champion_run.data.metrics['test_log_loss']:.4f})")
print(f"Challenger: {challenger_run.data.tags.get('mlflow.runName')}  (log_loss={challenger_run.data.metrics['test_log_loss']:.4f})")

Champion:   lr_baseline  (log_loss=0.2753)
Challenger: xgb_v1  (log_loss=0.2785)


## `mlflow.evaluate` — formal evaluation report

`mlflow.evaluate` runs a battery of classification metrics on the model, builds
explainability plots, and logs them all back to the run. Great for a production
sign-off step.

In [3]:
from sklearn.model_selection import train_test_split

FEATURE_COLS = [
    "distance_ft", "angle_deg", "distance_sq", "angle_sq",
    "is_high_danger", "rebound", "rush", "rebound_dist", "rush_dist",
    "period", "time_in_period_sec", "prev_event_sec",
    "st_wrist", "st_snap", "st_slap", "st_backhand", "st_tip", "st_wrap",
    "str_5v5", "str_5v4_pp", "str_4v5_pk", "str_4v4", "str_3v3_ot", "str_6v5_en", "str_5v6",
]

pdf = spark.table(FEATURES).select(*FEATURE_COLS, "goal").toPandas()
_, eval_df = train_test_split(pdf, test_size=0.20, random_state=42, stratify=pdf["goal"])
# Cast features to float64 so they line up with the model's input signature
_FEATS = [c for c in FEATURE_COLS if c != "goal"]
for col in _FEATS:
    eval_df[col] = eval_df[col].astype("float64")
print(f"Eval set: {len(eval_df):,} rows  positive rate: {eval_df['goal'].mean()*100:.2f}%")

Eval set: 10,000 rows  positive rate: 9.42%


In [4]:
champion_uri = f"runs:/{champion_run.info.run_id}/model"

# Manual holdout evaluation: our pyfunc returns probabilities, so we use
# probability-aware metrics (log_loss, Brier, ROC AUC) rather than mlflow.evaluate's
# default classification metrics (which expect class labels).
import mlflow.pyfunc
from sklearn.metrics import (
    log_loss, brier_score_loss, roc_auc_score, average_precision_score
)

model = mlflow.pyfunc.load_model(champion_uri)
X_eval = eval_df[[c for c in FEATURE_COLS if c != "goal"]]
y_eval = eval_df["goal"].astype(int).to_numpy()
proba = model.predict(X_eval)
if hasattr(proba, "values"): proba = proba.values.reshape(-1)

eval_metrics = {
    "holdout_log_loss": log_loss(y_eval, proba),
    "holdout_brier":    brier_score_loss(y_eval, proba),
    "holdout_roc_auc":  roc_auc_score(y_eval, proba),
    "holdout_pr_auc":   average_precision_score(y_eval, proba),
}
with mlflow.start_run(run_id=champion_run.info.run_id):
    for k, v in eval_metrics.items():
        mlflow.log_metric(k, v)

print("Holdout metrics on champion:")
for k, v in eval_metrics.items():
    print(f"  {k:25s} {v:.4f}")


/Users/alexander.booth/miniconda3/envs/demo-env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🏃 View run lr_baseline at: https://e2-demo-field-eng.cloud.databricks.com/ml/experiments/112419715816526/runs/81e2cbee212d4a69b7a6075d7178794d
🧪 View experiment at: https://e2-demo-field-eng.cloud.databricks.com/ml/experiments/112419715816526
Holdout metrics on champion:
  holdout_log_loss          0.2753
  holdout_brier             0.0775
  holdout_roc_auc           0.7470
  holdout_pr_auc            0.2730


## Register to Unity Catalog

`mlflow.register_model` against `databricks-uc` writes a model version under
`{catalog}.{schema}.{model_name}`. Each registration bumps the version number.

Aliases (`@champion`, `@challenger`, `@shadow`, etc.) are the modern replacement
for the legacy `Stage` API (`Staging` / `Production`).

In [5]:
champ_mv = mlflow.register_model(
    model_uri=f"runs:/{champion_run.info.run_id}/model",
    name=MODEL_NAME,
    tags={"source_run": champion_run.info.run_id,
          "model_family": champion_run.data.tags.get("model_family", ""),
          "intent": "production_candidate"},
)
print(f"Registered champion: {MODEL_NAME} v{champ_mv.version}")

chal_mv = mlflow.register_model(
    model_uri=f"runs:/{challenger_run.info.run_id}/model",
    name=MODEL_NAME,
    tags={"source_run": challenger_run.info.run_id,
          "model_family": challenger_run.data.tags.get("model_family", ""),
          "intent": "challenger"},
)
print(f"Registered challenger: {MODEL_NAME} v{chal_mv.version}")

Registered model 'alexander_booth.hockey_xg_mlflow.xg_model' already exists. Creating a new version of this model...


Uploading artifacts:   0%|          | 0/7 [00:00<?, ?it/s]

Uploading artifacts:  14%|█▍        | 1/7 [00:00<00:02,  2.59it/s]

Uploading artifacts:  14%|█▍        | 1/7 [00:00<00:02,  2.59it/s]

Uploading artifacts:  29%|██▊       | 2/7 [00:00<00:01,  2.59it/s]

...

Uploading artifacts:  86%|████████▌ | 6/7 [00:00<00:00,  6.38it/s]

Uploading artifacts: 100%|██████████| 7/7 [00:00<00:00,  6.38it/s]

Uploading artifacts: 100%|██████████| 7/7 [00:00<00:00,  8.18it/s]

Registered challenger: alexander_booth.hockey_xg_mlflow.xg_model v6


Created version '6' of model 'alexander_booth.hockey_xg_mlflow.xg_model'.


In [6]:
# Set aliases
client.set_registered_model_alias(MODEL_NAME, "champion",   champ_mv.version)
client.set_registered_model_alias(MODEL_NAME, "challenger", chal_mv.version)

# Set a UC model description
client.update_registered_model(
    name=MODEL_NAME,
    description=("Expected-goals (xG) classifier — predicts P(goal) for an "
                 "unblocked shot attempt given location, type, strength state, "
                 "and prior-event context. Trained on synthetic event data."),
)

# Add a version-level description on the champion
client.update_model_version(
    name=MODEL_NAME,
    version=champ_mv.version,
    description=f"Champion as of registration. Source run {champion_run.info.run_id}.",
)
print(f"\nAliases set:")
print(f"  {MODEL_NAME}@champion   -> v{champ_mv.version}")
print(f"  {MODEL_NAME}@challenger -> v{chal_mv.version}")


Aliases set:
  alexander_booth.hockey_xg_mlflow.xg_model@champion   -> v5
  alexander_booth.hockey_xg_mlflow.xg_model@challenger -> v6


In [7]:
# Confirm by reading back
mv = client.get_model_version_by_alias(MODEL_NAME, "champion")
print(f"@champion resolves to v{mv.version}, run_id={mv.run_id}")
print(f"Model URI for inference: models:/{MODEL_NAME}@champion")

@champion resolves to v5, run_id=81e2cbee212d4a69b7a6075d7178794d
Model URI for inference: models:/alexander_booth.hockey_xg_mlflow.xg_model@champion


**Next:** `05_batch_inference` loads `models:/{MODEL_NAME}@champion` and scores
a fresh holdout. Then `06_serving_endpoint` deploys the same alias to a Serving endpoint.